In [ ]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

In [ ]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!git clone https://github.com/Omid-Nejati/MedViTV2.git

In [ ]:
!pip install natten==0.17.3+torch250cu124 -f https://shi-labs.com/natten/wheels/ --trusted-host shi-labs.com

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return image, localiser, label


train_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), transform=train_transform)
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), transform=test_transform)
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), transform=test_transform)

labels = df_train['label'].values.astype(int)

class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_ds, batch_size=12, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=12, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)

n_labels = df_train['label'].nunique()





In [ ]:
%cd /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/MedViTV2-main
!pip install -r requirements.txt
from MedViT import MedViT

In [ ]:



class DualBranchMedViT(nn.Module):
    def __init__(self, num_classes, pretrained_weights=None, **kwargs):
        super().__init__()
        
        # Two separate MedViT encoders (or shared weights — your choice)
        self.branch_image = MedViT(num_classes=num_classes, **kwargs)
        self.branch_localizer = MedViT(num_classes=num_classes, **kwargs)
        
        # Remove both classification heads — we'll fuse before classifying
        final_dim = self.branch_image.proj_head[0].in_features  # e.g. 512
        self.branch_image.proj_head = nn.Identity()
        self.branch_localizer.proj_head = nn.Identity()
        
        # Fusion + classification head
        self.classifier = nn.Sequential(
            nn.Linear(final_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        
        if pretrained_weights:
            state_dict = torch.load(pretrained_weights, map_location='cpu')
            self.branch_image.load_state_dict(state_dict, strict=False)
            self.branch_localizer.load_state_dict(state_dict, strict=False)

    def forward(self, image, localizer):
        feat_img = self.branch_image(image)        # (B, final_dim)
        feat_loc = self.branch_localizer(localizer) # (B, final_dim)
        fused = torch.cat([feat_img, feat_loc], dim=1)  # (B, final_dim*2)
        return self.classifier(fused)